<a href="https://colab.research.google.com/github/Rasheena-Arimbrathodi/BBC-web-scraping-using-requests-and-beautiful-soup/blob/main/ppp_pes2o_science_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dataset citation

In [ ]:
'''@techreport{peS2o,
    author = {Luca Soldaini and Kyle Lo},
    year = 2023,
    title = {{peS2o (Pretraining Efficiently on S2ORC) Dataset}},
    institution = {{Allen Institute for AI}},
    note = {ODC-By, \url{https://github.com/allenai/pes2o}}
}'''


SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 210-211: truncated \uXXXX escape (ipython-input-929516296.py, line 1)

In [ ]:
!pip install requests pandas pyarrow zstandard

In [ ]:
!pip install huggingface-hub zstandard pandas pyarrow

In [ ]:
#dowloading valid sets from huggingface pes2o-validation and uploading in the drive

from huggingface_hub import hf_hub_download
import zstandard
import pandas as pd

# 1. Define the repository and file details
repo_id = "allenai/peS2o"
file_in_repo = "data/v3/train-0000-of-0136.zst"
output_filename = "output_data.jsonl"

print(f"Downloading '{file_in_repo}' from dataset '{repo_id}'...")

try:
    # 2. Use hf_hub_download to safely get the file path
    # This correctly handles Git LFS and caches the download for future runs.
    downloaded_file_path = hf_hub_download(
        repo_id=repo_id,
        filename=file_in_repo,
        repo_type="dataset"  # Specify that it's a dataset
    )
    print(f"✅ Download complete. File cached at: {downloaded_file_path}")

    # 3. Decompress the file and read it directly into pandas
    print("Decompressing and reading Parquet data...")
    dctx = zstandard.ZstdDecompressor()
    with open(downloaded_file_path, 'rb') as compressed_file:
        with dctx.stream_reader(compressed_file) as reader:
            # The reader stream is passed directly to pandas
            df = pd.read_parquet(reader)
    print("✅ Data loaded successfully.")

    # 4. Save the DataFrame to a JSON Lines (.jsonl) file
    print(f"Saving data to '{output_filename}'...")
    df.to_json(output_filename, orient='records', lines=True, force_ascii=False)

    print(f"🎉 Success! Data has been saved to '{output_filename}'.")

except Exception as e:
    print(f"An unexpected error occurred: {e}")

data/v3/train-0000-of-0136.zst:   0%|          | 0.00/645M [00:00<?, ?B/s]

✅ Download complete. File cached at: /root/.cache/huggingface/hub/datasets--allenai--peS2o/snapshots/636a503e44a3ca1b58e01fb61eab0825cd574de0/data/v3/train-0000-of-0136.zst
Decompressing and reading Parquet data...
An unexpected error occurred: zstd decompression streams cannot be seeked with SEEK_END


In [ ]:
#Analysing one zst file's output.
from google.colab import drive
import os
import zstandard as zstd
import io
import json

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Path to your validation folder
folder_path = "/content/drive/MyDrive/peS2o_v3_valid/"
first_file = os.path.join(folder_path, "valid-0000-of-0060.zst")

# 3. Output jsonl file (raw copy)
output_file = "/content/drive/MyDrive/peS2o_valid_0000_raw.jsonl"

# 4. Read the zst file
with open(first_file, "rb") as f:
    dctx = zstd.ZstdDecompressor()
    stream_reader = dctx.stream_reader(f)
    text_stream = io.TextIOWrapper(stream_reader, encoding="utf-8")

    # Write raw lines to jsonl
    with open(output_file, "w", encoding="utf-8") as writer:
        for i, line in enumerate(text_stream):
            writer.write(line)  # store as-is
            if i < 3:  # print first 3 docs for inspection
                print(json.loads(line))
            if i == 10:  # stop after 10 docs printed to avoid overload
                break

print("✅ Raw JSONL copy saved at:", output_file)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
{'id': '272452128', 'source': 'pes2o/s2ag', 'version': 'v3-fos-license', 'added': '2024-09-08T15:15:59.703Z', 'created': '2024-09-06T00:00:00.000', 'text': 'Modified Handball in Physical Education: Investigating Opportunities for Inclusion and Relatedness\n\nThis paper addresses the challenge of assessing relatedness and functional interdependence through connecting passes within invasion games, which may offer valuable pedagogical insights into gameplay for accessibility and inclusiveness. Hence, the purpose of this paper is twofold. Firstly, it presents preliminary work on the methodology for computing open passing lanes and derived metrics, integrating spatiotemporal data analysis with event data. Secondly, using a within-subject design, it investigates how modified handball games influence game play opportunities. Data were collected during handball match

In [ ]:
#Found more than one paragraph in each line. So seperating each paragraph and giving a uniques id. We need metadata with id, source and pargraph_id(our unique id) and the paragraph in each line of jsonl.
#Checking for the few lines.
from google.colab import drive
import os
import json
import zstandard as zstd
import io

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Path to your validation folder in Drive
folder_path = "/content/drive/MyDrive/peS2o_v3_valid"   # change if needed
first_file = os.path.join(folder_path, "valid-0000-of-0060.zst")

# 3. Output JSONL file (test with first 3 docs)
output_file = "/content/drive/MyDrive/peS2o_valid_3docs.jsonl"

para_counter = 1  # global paragraph counter

with open(first_file, "rb") as f:
    dctx = zstd.ZstdDecompressor()
    stream_reader = dctx.stream_reader(f)
    text_stream = io.TextIOWrapper(stream_reader, encoding="utf-8")

    with open(output_file, "w", encoding="utf-8") as writer:
        for i, line in enumerate(text_stream):
            if i >= 3:  # stop after 3 docs
                break
            try:
                doc = json.loads(line)

                # Split into paragraphs
                paragraphs = [p.strip() for p in doc.get("text", "").split("\n\n") if p.strip()]

                for p in paragraphs:
                    out = {
                        "metadata": {
                            "para_id": para_counter,
                            "doc_id": doc.get("id"),
                            "source": doc.get("source")
                        },
                        "paragraph": p
                    }

                    writer.write(json.dumps(out, ensure_ascii=False) + "\n")
                    print(json.dumps(out, indent=2, ensure_ascii=False))  # print sample

                    para_counter += 1

            except Exception as e:
                print(f"⚠️ Skipping line due to error: {e}")

print("✅ Sample JSONL with para_id saved at:", output_file)


Mounted at /content/drive
{
  "metadata": {
    "para_id": 1,
    "doc_id": "272452128",
    "source": "pes2o/s2ag"
  },
  "paragraph": "Modified Handball in Physical Education: Investigating Opportunities for Inclusion and Relatedness"
}
{
  "metadata": {
    "para_id": 2,
    "doc_id": "272452128",
    "source": "pes2o/s2ag"
  },
  "paragraph": "This paper addresses the challenge of assessing relatedness and functional interdependence through connecting passes within invasion games, which may offer valuable pedagogical insights into gameplay for accessibility and inclusiveness. Hence, the purpose of this paper is twofold. Firstly, it presents preliminary work on the methodology for computing open passing lanes and derived metrics, integrating spatiotemporal data analysis with event data. Secondly, using a within-subject design, it investigates how modified handball games influence game play opportunities. Data were collected during handball matches in a pre-teens Physical Education (

In [ ]:
# Extracting the entire zst files 60/60 and keeping it in the jsnol file by zipping it.
from google.colab import drive
import os
import json
import zstandard as zstd
import io
import zipfile

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Path to your validation folder in Drive
folder_path = "/content/drive/MyDrive/peS2o_v3_valid"   # change if needed
output_file = "/content/drive/MyDrive/peS2o_valid_all.jsonl"
zipped_file_path = "/content/drive/MyDrive/peS2o_valid_all.zip"

para_counter = 1  # global counter for unique para_id

# 3. Function to process each .zst file
def process_zst_file(file_path, writer):
    global para_counter
    with open(file_path, "rb") as f:
        dctx = zstd.ZstdDecompressor()
        stream_reader = dctx.stream_reader(f)
        text_stream = io.TextIOWrapper(stream_reader, encoding="utf-8")

        for line in text_stream:
            try:
                doc = json.loads(line)

                # Split into paragraphs
                paragraphs = [p.strip() for p in doc.get("text", "").split("\n\n") if p.strip()]

                for p in paragraphs:
                    out = {
                        "metadata": {
                            "para_id": para_counter,
                            "doc_id": doc.get("id"),
                            "source": doc.get("source")
                        },
                        "paragraph": p
                    }

                    writer.write(json.dumps(out, ensure_ascii=False) + "\n")
                    para_counter += 1

            except Exception as e:
                print(f"⚠️ Skipping line in {file_path} due to error: {e}")

# 4. Collect all validation files
val_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith(".zst")]
val_files.sort()

# 5. Process and merge into one JSONL
with open(output_file, "w", encoding="utf-8") as writer:
    for fpath in val_files:
        print(f"📂 Processing {fpath} ...")
        process_zst_file(fpath, writer)

print(f"✅ Finished! JSONL saved at: {output_file}")

# 6. Zip the JSONL file
if os.path.exists(output_file):
    with zipfile.ZipFile(zipped_file_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(output_file, os.path.basename(output_file))
    print(f"✅ File zipped successfully: {zipped_file_path}")
else:
    print(f"❌ Error: The file {output_file} does not exist.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📂 Processing /content/drive/MyDrive/peS2o_v3_valid/valid-0000-of-0060.zst ...
📂 Processing /content/drive/MyDrive/peS2o_v3_valid/valid-0001-of-0060.zst ...
📂 Processing /content/drive/MyDrive/peS2o_v3_valid/valid-0002-of-0060.zst ...
📂 Processing /content/drive/MyDrive/peS2o_v3_valid/valid-0003-of-0060.zst ...
📂 Processing /content/drive/MyDrive/peS2o_v3_valid/valid-0004-of-0060.zst ...
📂 Processing /content/drive/MyDrive/peS2o_v3_valid/valid-0005-of-0060.zst ...
📂 Processing /content/drive/MyDrive/peS2o_v3_valid/valid-0006-of-0060.zst ...
📂 Processing /content/drive/MyDrive/peS2o_v3_valid/valid-0007-of-0060.zst ...
📂 Processing /content/drive/MyDrive/peS2o_v3_valid/valid-0008-of-0060.zst ...
📂 Processing /content/drive/MyDrive/peS2o_v3_valid/valid-0009-of-0060.zst ...
📂 Processing /content/drive/MyDrive/peS2o_v3_valid/valid-0010-of-0060.zst ...
📂 Processing 

In [ ]:
from google.colab import drive
import os
import json
import zstandard as zstd
import io
import zipfile

# 1. Mount Google Drive
drive.mount('/content/drive')
jsonl_path = "/content/drive/MyDrive/peS2o_valid_all.jsonl"

last_line = None
with open(jsonl_path, "r", encoding="utf-8") as f:
    for line in f:   # iterates efficiently line by line
        last_line = line

if last_line:
    doc = json.loads(last_line)
    print("📌 Last para_id:", doc["metadata"]["para_id"])
else:
    print("❌ File is empty")


Mounted at /content/drive
📌 Last para_id: 2043044


1. Hindi
2. Malayalam
3. Urdu
4. Tamil
5. Telugu
6. Marati
7. Bengali
8. Odiya
9. Kannada
10. Gujarati
11. Assamese
12. Panjabi